# 01. Análisis Exploratorio de Datos (EDA) y Embeddings GloVe

Este notebook se centra en la carga, limpieza y análisis inicial del dataset **GoEmotions**. Exploraremos la distribución de las emociones de Ekman y utilizaremos técnicas clásicas como **TF-IDF** y **GloVe** para entender la representación semántica de los comentarios antes de aplicar modelos de Deep Learning.

In [ ]:
import re
import random
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.manifold import TSNE
from datasets import load_dataset, DatasetDict
import gensim.downloader as api

# Configuración estética
plt.style.use('default')
sns.set_palette("tab10")

### 1. Definición de la Función de Mapeo a Ekman

El dataset original contiene 28 emociones detalladas. Para facilitar el aprendizaje del modelo y la interpretación de resultados, agrupamos estas emociones en las 7 categorías universales de Paul Ekman.

In [ ]:
def map_fine_to_ekman(dataset):
    """
    Transforma las 28 emociones originales en las 7 categorías de Ekman.
    
    Lógica de la función:
    1. Define un diccionario de mapeo entre categorías de Ekman y etiquetas originales.
    2. Invierte el mapeo para crear un diccionario de búsqueda rápida.
    3. Define una función interna 'convert' que procesa cada fila, contando las ocurrencias 
       de emociones de Ekman y seleccionando la predominante en caso de multi-etiquetado.
    """
    ekman2fine_mapping = {
        "anger": ["anger", "annoyance", "disapproval"],
        "disgust": ["disgust"],
        "fear": ["fear", "nervousness"],
        "joy": ["joy", "amusement", "approval", "excitement", "gratitude",
                "love", "optimism", "relief", "pride", "admiration", "desire", "caring"],
        "sadness": ["sadness", "disappointment", "embarrassment", "grief", "remorse"],
        "surprise": ["surprise", "realization", "confusion", "curiosity"],
        "neutral": ["neutral"],
    }

    ekman2label = {cat: i for i, cat in enumerate(sorted(ekman2fine_mapping.keys()))}
    fine_to_ekman = {fine: ekman_cat for ekman_cat, fine_list in ekman2fine_mapping.items() for fine in fine_list}
    fine_labels = dataset["train"].features["labels"].feature.names

    def convert(example):
        fine_emos = [fine_labels[i] for i in example["labels"]]
        ekman_emos = [fine_to_ekman.get(e, "other") for e in fine_emos]
        counts = Counter(ekman_emos)
        # Elegir la más frecuente o neutral por defecto si no hay coincidencia
        chosen = max(counts.items(), key=lambda x: x[1])[0] if counts else "neutral"
        return {"text": example["text"], 'class': chosen, 'labels': ekman2label[chosen]}

    new_splits = {}
    for split in dataset.keys():
        new_splits[split] = dataset[split].map(convert, remove_columns=dataset[split].column_names)
    return DatasetDict(new_splits)

### 2. Carga del Dataset y Análisis Estadístico

Cargamos los datos directamente desde Hugging Face y aplicamos el mapeo. Analizamos el volumen de datos en cada conjunto para asegurar la integridad de los splits.

In [ ]:
# Carga del dataset original
raw_dataset = load_dataset("go_emotions", "simplified")

# Mapeo a categorías de Ekman
ekman_dataset = map_fine_to_ekman(raw_dataset)

# Mostrar tamaños de los conjuntos para verificar la carga
for split in ekman_dataset.keys():
    print(f"Conjunto {split}: {len(ekman_dataset[split])} instancias")

### 3. Visualización de la Distribución de Clases

Es vital entender el balance del dataset. Un desequilibrio notable puede sesgar los modelos de Deep Learning hacia las clases mayoritarias.

In [ ]:
# Cálculo de porcentajes para visualización comparativa
data_list = []
for split in ["train", "validation", "test"]:
    total = len(ekman_dataset[split])
    counts = Counter(ekman_dataset[split]["class"])
    for emotion, count in counts.items():
        data_list.append({"Emotion": emotion, "Percentage": (count/total)*100, "Split": split})

df_plot = pd.DataFrame(data_list)

plt.figure(figsize=(12, 6))
sns.barplot(data=df_plot, x="Emotion", y="Percentage", hue="Split")
plt.title("Distribución de Categorías de Ekman por Split")
plt.ylabel("Porcentaje (%)")
plt.show()

# Análisis numérico del desbalanceo
df_train = ekman_dataset["train"].to_pandas()
counts = df_train["class"].value_counts()
print(f"\nRatio Clase Mayoritaria ({counts.index[0]}) vs Minoritaria ({counts.index[-1]}): {counts.iloc[0]/counts.iloc[-1]:.2f}x")

### 4. Extracción de Palabras Clave (TF-IDF)

Utilizamos la técnica estadística TF-IDF para identificar qué términos son más distintivos para cada emoción. Esto nos permite validar si las etiquetas humanas guardan coherencia semántica.

In [ ]:
train_texts = ekman_dataset["train"]["text"]
train_labels = ekman_dataset["train"]["class"]

# TfidfVectorizer convierte texto en vectores numéricos ponderados por importancia
vectorizer = TfidfVectorizer(stop_words="english", min_df=5)
X_tfidf = vectorizer.fit_transform(train_texts)
feature_names = np.array(vectorizer.get_feature_names_out())

print("Top 5 palabras por emoción (TF-IDF):")
for emotion in sorted(set(train_labels)):
    # Seleccionamos filas correspondientes a la emoción actual
    indices = np.where(np.array(train_labels) == emotion)[0]
    # Calculamos la puntuación TF-IDF media de cada palabra para esa emoción
    mean_tfidf = np.asarray(X_tfidf[indices].mean(axis=0)).flatten()
    top_indices = mean_tfidf.argsort()[::-1][:5]
    print(f"[{emotion.upper()}]: {feature_names[top_indices]}")

### 5. Representación Semántica con GloVe y t-SNE

Transformamos los comentarios en vectores densos promediando los embeddings **GloVe** de sus palabras. Luego, utilizamos **t-SNE** para proyectar estos datos de alta dimensión (50D) en un plano 2D y observar si existen agrupamientos naturales por emoción.

In [ ]:
print("Cargando embeddings GloVe (glove-wiki-gigaword-50)...")
glove = api.load("glove-wiki-gigaword-50")

# Expresión regular para tokenización básica
TOKEN_RE = re.compile(r"\b\w+\b", flags=re.UNICODE)
def tokenize(text): return [t.lower() for t in TOKEN_RE.findall(text)]

val_texts = ekman_dataset["validation"]["text"]
val_labels = ekman_dataset["validation"]["class"]
X_val_emb, y_val_clean = [], []

for text, label in zip(val_texts, val_labels):
    # Buscamos el vector de cada palabra en el modelo GloVe
    vectors = [glove[w] for w in tokenize(text) if w in glove]
    if vectors:
        # Promediamos los vectores de la frase para generar un embedding único
        X_val_emb.append(np.mean(vectors, axis=0))
        y_val_clean.append(label)

# Configuración de t-SNE para reducción de dimensionalidad
tsne = TSNE(n_components=2, random_state=42)
X_embedded = tsne.fit_transform(np.array(X_val_emb))

plt.figure(figsize=(10, 8))
sns.scatterplot(x=X_embedded[:,0], y=X_embedded[:,1], hue=y_val_clean, alpha=0.6, palette="tab10", s=20)
plt.title("Visualización t-SNE de Embeddings GloVe (Promedio)")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Conclusiones Técnicas

1. **Desbalance de Clases:** Se observa un desequilibrio significativo en el conjunto de datos. Las categorías `joy` y `neutral` representan la gran mayoría de los ejemplos, lo que requerirá métricas de evaluación robustas (como F1-Score Macro) en los modelos posteriores.
2. **Calidad Semántica:** Los términos extraídos mediante TF-IDF muestran una alta correlación con las etiquetas (ej. 'scared' para miedo, 'love' para alegría), validando la coherencia del dataset.
3. **Necesidad de Contexto:** El solapamiento observado en el análisis t-SNE con GloVe sugiere que el uso de embeddings estáticos promediados no captura la complejidad emocional. Esto motiva la transición hacia arquitecturas basadas en **Atención** y **Transformers** para capturar relaciones contextuales más precisas.